# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:
# Write your code below.
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Get the price data path
PRICE_DATA = os.getenv("PRICE_DATA")

if PRICE_DATA is None:
    raise ValueError(
        "PRICE_DATA is not set. Please run "
        "01_materials/labs/2_data_engineering.ipynb first."
    )

print("PRICE_DATA:", PRICE_DATA)


In [ ]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.


import os
from dotenv import load_dotenv
from glob import glob

# Load environment variables
load_dotenv()

# Get the PRICE_DATA directory
price_data = os.getenv("PRICE_DATA")

# Find all Parquet files in the directory
parquet_files = glob(os.path.join(price_data, "*.parquet"))

# Display the file paths
print(parquet_files)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
import dask.dataframe as dd

# Load the Parquet files using Dask
dd_feat = dd.read_parquet(parquet_files)

# Sort by ticker and date to ensure correct lag calculations
dd_feat = dd_feat.map_partitions(
    lambda df: df.sort_values(["Ticker", "Date"])
)

# Add lag features for each ticker
dd_feat["Close_lag_1"] = dd_feat.groupby("Ticker")["Close"].shift(1)
dd_feat["Close_lag_2"] = dd_feat.groupby("Ticker")["Close"].shift(2)

dd_feat["Adj_Close_lag_1"] = dd_feat.groupby("Ticker")["Adj_Close"].shift(1)
dd_feat["Adj_Close_lag_2"] = dd_feat.groupby("Ticker")["Adj_Close"].shift(2)

# Calculate returns based on Close
dd_feat["returns"] = (dd_feat["Close"] / dd_feat["Close_lag_1"]) - 1

# Calculate the daily high-low range
dd_feat["hi_lo_range"] = dd_feat["High"] - dd_feat["Low"]


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

# Convert Dask DataFrame to pandas
df_feat = dd_feat.compute()

# Sort by ticker and date
df_feat = df_feat.sort_values(["Ticker", "Date"])


df_feat["returns_ma_10"] = (
    df_feat.groupby("Ticker")["returns"]
    .transform(lambda x: x.rolling(10).mean())
)


df_feat.head()

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return? No
+ Would it have been better to do it in Dask? Why? Yes it would have improve the processing time

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.